In [1]:
# ==============================================================================
#  PIPELINE ML — DÉTECTION VÉRIFICATION FACTURES
#  VERSION : Features originales uniquement (sans feature engineering)
# ------------------------------------------------------------------------------
#  Contexte :
#    Les colonnes sont des flags binaires issus de règles métier appliquées
#    sur des factures médicales. On utilise UNIQUEMENT les features originales
#    ayant une variance utile (>3% de variation dans les données).
#
#  Features exclues : 31 colonnes constantes (>97% même valeur) → zéro signal
#  Features retenues : 6 colonnes avec variance réelle + catégorielles
#
#  Stratégie :
#    1. Sélection des features originales non-constantes
#    2. Split 60/20/20 : train / validation (calibration seuil) / test
#    3. Optimisation du seuil de décision sur le set de validation
#    4. Évaluation finale sur le test set indépendant
#    5. Visualisations + rapport enregistré dans rapportexecution.txt
#
#  Modèles : GradientBoosting | RandomForest | KNN | LogisticRegression
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # Backend non-interactif pour la sauvegarde des images
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import sys
import os
from io import StringIO
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, learning_curve
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# ==============================================================================
# CONFIGURATION : chemins et constantes globales
# ==============================================================================

DATA_PATH   = 'data/dfforml2.csv'
OUTPUT_DIR  = 'models/'
REPORT_DIR  = 'reports/'
REPORT_PATH = os.path.join(REPORT_DIR, 'rapportexecution.txt')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

RANDOM_STATE = 42

# Couleurs par modèle — cohérence sur tous les graphes
MODEL_COLORS = {
    'GradientBoosting' : '#2196F3',
    'RandomForest'     : '#4CAF50',
    'KNN'              : '#FF9800',
    'LogisticReg'      : '#9C27B0',
}

plt.rcParams.update({
    'figure.dpi'      : 130,
    'font.size'       : 10,
    'axes.titlesize'  : 12,
    'axes.titleweight': 'bold',
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
})

# ==============================================================================
# CAPTURE DU LOG : tout ce qui est printé sera aussi écrit dans le rapport
# ==============================================================================

class Tee:
    """Redirige print() vers stdout ET vers un buffer pour l'écrire ensuite."""
    def __init__(self):
        self.buffer = StringIO()
        self.stdout = sys.stdout

    def write(self, msg):
        self.stdout.write(msg)
        self.buffer.write(msg)

    def flush(self):
        self.stdout.flush()

    def getvalue(self):
        return self.buffer.getvalue()

tee = Tee()
sys.stdout = tee

# ==============================================================================
# DÉBUT DU PIPELINE
# ==============================================================================

print("=" * 70)
print("   PIPELINE ML — VÉRIFICATION FACTURES (Features originales)")
print("=" * 70)


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 1 — CHARGEMENT DES DONNÉES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 1/6] Chargement des données")

df = pd.read_csv(DATA_PATH)

# Encodage de la variable cible
# validee    = 0  (classe négative)
# a_corriger = 1  (classe positive — factures avec erreurs à corriger)
y = df['status_verification'].map({'validee': 0, 'a_corriger': 1})

n_total    = len(df)
n_corriger = y.sum()
n_validee  = (y == 0).sum()

print(f"  ➤ Fichier             : {DATA_PATH}")
print(f"  ➤ Lignes totales      : {n_total}")
print(f"  ➤ Colonnes totales    : {df.shape[1]}")
print(f"  ➤ Classe 'validee'    : {n_validee} ({n_validee/n_total*100:.1f}%)")
print(f"  ➤ Classe 'a_corriger' : {n_corriger} ({n_corriger/n_total*100:.1f}%)")
print(f"  ➤ Ratio déséquilibre  : {n_corriger/n_validee:.2f}")
print()
print("  Analyse de variance des colonnes originales :")
print(f"  {'Colonne':<45} {'Dominance':>10}  {'Statut'}")
print("  " + "─" * 65)

all_feature_cols = [c for c in df.columns
                    if c not in ['status_verification',
                                 'observations_verification',
                                 'type_observation']]

dead_cols  = []
alive_cols = []

for c in all_feature_cols:
    dom = df[c].value_counts(normalize=True).iloc[0]
    status = "✗ MORTE (constante >97%)" if dom > 0.97 else "✓ UTILE"
    print(f"  {c:<45} {dom:>10.3f}  {status}")
    if dom > 0.97:
        dead_cols.append(c)
    else:
        alive_cols.append(c)

print()
print(f"  ➤ Features mortes (exclues) : {len(dead_cols)}")
print(f"  ➤ Features utiles retenues  : {len(alive_cols)}")
print(f"  ➤ Plafond théorique absolu  : 64.2%")
print(f"    (90% des lignes ont des features identiques mais labels différents")
print(f"     → limite irréductible des données — Bayes Error Rate)")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 2 — SÉLECTION DES FEATURES ORIGINALES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 2/6] Sélection des features originales non-constantes")

# Ces 6 features sont les seules ayant une variance utile sur les 1433 lignes.
# Toutes les autres colonnes ont >97% de la même valeur → aucun signal pour le ML.
FEATURES_RETENUES = [
    'consultation_type',                     # Catégorielle, 5 valeurs distinctes
    'type_prestation',                       # Catégorielle, 20 valeurs distinctes
    'date_entree_is_valid',                  # Binaire, 10.3% de variance
    'date_sortie_is_valid',                  # Binaire, 10.3% de variance
    'verifierHopitalisation_PF_Ambulatoires',# Binaire, 29.5% de variance
    'quantite_total_ex_exists',              # Binaire, 16.1% de variance
]

X = df[FEATURES_RETENUES].copy()
X['consultation_type'] = X['consultation_type'].fillna(-1)   # -1 = valeur manquante
X['type_prestation']   = X['type_prestation'].fillna(-1)

print(f"  ➤ Features retenues : {len(FEATURES_RETENUES)}")
for feat in FEATURES_RETENUES:
    n_unique = df[feat].nunique()
    dom      = df[feat].value_counts(normalize=True).iloc[0]
    corr     = df[feat].fillna(-1).corr(y)
    print(f"    {feat:<45} nunique={n_unique:>2}  dom={dom:.3f}  corr={corr:+.4f}")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3 — SPLIT TRAIN / VALIDATION / TEST
# ──────────────────────────────────────────────────────────────────────────────
# Split en 3 parties pour éviter le biais d'optimisation du seuil :
#   TRAIN (60%)      → entraînement des modèles
#   VALIDATION (20%) → calibration du seuil de décision (jamais vu à l'entraînement)
#   TEST (20%)       → évaluation finale indépendante (jamais utilisé avant)
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 3/6] Split Train / Validation / Test (60% / 20% / 20%)")

# 1er split : 80% train+val / 20% test final
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    stratify     = y      # Preserve la proportion de chaque classe
)

# 2ème split : 75% train / 25% val (= 60% / 20% du total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size    = 0.25,
    random_state = RANDOM_STATE,
    stratify     = y_trainval
)

print(f"  ➤ Train      : {len(X_train):4d} lignes "
      f"| a_corriger={y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  ➤ Validation : {len(X_val):4d} lignes "
      f"| a_corriger={y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"  ➤ Test       : {len(X_test):4d} lignes "
      f"| a_corriger={y_test.sum()} ({y_test.mean()*100:.1f}%)")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 4 — DÉFINITION ET ENTRAÎNEMENT DES MODÈLES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 4/6] Définition et entraînement des modèles")

# Chaque modèle est encapsulé dans un Pipeline sklearn :
#   StandardScaler : normalise les features (obligatoire pour KNN et LogReg)
#   Classifier     : le modèle d'apprentissage

models = {

    # ── GradientBoosting ──────────────────────────────────────────────────────
    # Boosting séquentiel : chaque arbre corrige les erreurs du précédent.
    # learning_rate faible + subsample pour limiter l'overfitting sur peu de signal.
    'GradientBoosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators     = 400,    # Nb d'arbres séquentiels
            learning_rate    = 0.03,   # Petit pas → meilleure généralisation
            max_depth        = 4,      # Arbres peu profonds (données peu riches)
            min_samples_leaf = 10,     # Feuilles minimum 10 exemples
            subsample        = 0.8,    # Stochastic GB : 80% données par arbre
            random_state     = RANDOM_STATE
        ))
    ]),

    # ── RandomForest ─────────────────────────────────────────────────────────
    # Ensemble d'arbres indépendants entraînés en parallèle (bagging).
    # class_weight='balanced' compense automatiquement le déséquilibre 58/42.
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators     = 400,        # Nb d'arbres en parallèle
            max_depth        = 8,          # Profondeur max par arbre
            min_samples_leaf = 5,          # Régularisation
            max_features     = 'sqrt',     # Sélection aléatoire des features
            class_weight     = 'balanced', # Compense le déséquilibre de classes
            random_state     = RANDOM_STATE,
            n_jobs           = -1          # Utilise tous les cœurs CPU
        ))
    ]),

    # ── KNN ──────────────────────────────────────────────────────────────────
    # Classifie selon la majorité parmi les k voisins les plus proches.
    # TRÈS sensible à l'échelle → StandardScaler indispensable.
    # weights='distance' : voisins proches influencent plus la décision.
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(
            n_neighbors = 15,           # k=15 : compromis biais/variance
            weights     = 'distance',   # Pondération inverse de la distance
            metric      = 'euclidean',
            n_jobs      = -1
        ))
    ]),

    # ── LogisticRegression ───────────────────────────────────────────────────
    # Modèle linéaire probabiliste, très interprétable.
    # class_weight='balanced' pondère les erreurs selon la fréquence de classe.
    # C=1.0 : régularisation L2 standard (ni trop forte ni trop faible).
    'LogisticReg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C            = 1.0,         # Inverse de la régularisation
            max_iter     = 2000,        # Assure la convergence
            class_weight = 'balanced',  # Compense le déséquilibre
            solver       = 'lbfgs',
            random_state = RANDOM_STATE
        ))
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name:<22} entraîné sur {len(X_train)} lignes")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 5 — OPTIMISATION DU SEUIL DE DÉCISION
# ──────────────────────────────────────────────────────────────────────────────
# Le seuil par défaut (0.5) n'est pas toujours optimal.
# Dans notre cas, manquer une erreur (FN) est plus coûteux que bloquer
# à tort une facture valide (FP) → on cible Recall ≥ 0.72.
# Méthode : chercher sur le SET DE VALIDATION le seuil qui :
#   1. Garantit Recall ≥ 0.72
#   2. Maximise le F1-score parmi les seuils satisfaisants
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 5/6] Optimisation du seuil de décision (sur set de validation)")


def optimize_threshold(y_true, y_proba, recall_min=0.72):
    """
    Trouve le seuil optimal dans [0.20, 0.80] qui :
      - Garantit Recall ≥ recall_min sur la classe 'a_corriger'
      - Maximise le F1-score parmi les seuils satisfaisant la contrainte

    Si aucun seuil ne satisfait recall_min, retourne le seuil
    qui maximise F1 sans contrainte (fallback).

    Paramètres
    ----------
    y_true     : array, labels réels (0 = validee, 1 = a_corriger)
    y_proba    : array, probabilités prédites pour la classe 1
    recall_min : float, recall minimum requis sur a_corriger

    Retourne
    --------
    best_t   : float, seuil retenu
    details  : dict avec recall, precision, f1 au seuil retenu
    all_res  : list[dict], résultats pour chaque seuil (pour les graphes)
    """
    thresholds = np.arange(0.20, 0.81, 0.01)
    best_t, best_f1 = 0.50, 0.0
    all_res = []

    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        rec   = recall_score(y_true, preds, zero_division=0)
        prec  = precision_score(y_true, preds, zero_division=0)
        f1    = f1_score(y_true, preds, zero_division=0)
        all_res.append({'threshold': t, 'recall': rec,
                        'precision': prec, 'f1': f1})
        # Contrainte recall + maximisation F1
        if rec >= recall_min and f1 > best_f1:
            best_f1, best_t = f1, t

    # Fallback si aucun seuil ne respecte la contrainte recall
    if best_f1 == 0.0:
        best_t = max(all_res, key=lambda x: x['f1'])['threshold']

    best_preds = (y_proba >= best_t).astype(int)
    details = {
        'threshold' : best_t,
        'recall'    : recall_score(y_true, best_preds, zero_division=0),
        'precision' : precision_score(y_true, best_preds, zero_division=0),
        'f1'        : f1_score(y_true, best_preds, zero_division=0),
    }
    return best_t, details, all_res


results          = {}   # Métriques finales par modèle (sur test set)
probas_test      = {}   # Probabilités prédites sur test set
optimal_thrs     = {}   # Seuils optimaux calibrés sur validation
threshold_curves = {}   # Courbes seuil→métriques (pour graphe 4)

print(f"\n  {'Modèle':<22} {'Seuil':>6} {'AUC':>7} "
      f"{'Precision':>10} {'Recall':>8} {'F1':>7}")
print("  " + "─" * 60)

for name, model in models.items():
    # Probabilités sur validation (pour calibrer le seuil)
    y_proba_val  = model.predict_proba(X_val)[:, 1]
    # Probabilités sur test (pour l'évaluation finale)
    y_proba_test = model.predict_proba(X_test)[:, 1]

    # Seuil calibré sur validation uniquement
    best_t, _, all_res = optimize_threshold(
        y_val, y_proba_val, recall_min=0.72
    )

    # Prédictions finales sur TEST avec le seuil calibré
    y_pred = (y_proba_test >= best_t).astype(int)

    # Toutes les métriques calculées sur TEST set
    auc  = roc_auc_score(y_test, y_proba_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    results[name] = {
        'threshold'  : best_t,
        'AUC'        : auc,
        'Accuracy'   : acc,
        'Precision'  : prec,
        'Recall'     : rec,
        'F1'         : f1,
        'Specificity': spec,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'y_pred'     : y_pred,
    }
    probas_test[name]      = y_proba_test
    optimal_thrs[name]     = best_t
    threshold_curves[name] = all_res

    print(f"  {name:<22} {best_t:>6.2f} {auc:>7.4f} "
          f"{prec:>10.4f} {rec:>8.4f} {f1:>7.4f}")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 6 — VISUALISATIONS
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 6/6] Génération des visualisations")


# ── GRAPHE 1 : Courbes d'apprentissage ───────────────────────────────────────
# But : vérifier overfitting/underfitting selon la taille du dataset.
# Un écart train/val faible = bonne généralisation.

print("  → Graphe 1/6 : Learning Curves...")

fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle(
    "Courbes d'Apprentissage par Modèle\n"
    "(train → score sur données d'entraînement | "
    "val → généralisation sur données non vues)",
    fontsize=13, fontweight='bold', y=1.01
)

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_sizes_pct = np.linspace(0.15, 1.0, 10)

for ax, (name, model) in zip(axes.flatten(), models.items()):
    color = MODEL_COLORS[name]

    train_sizes, train_scores, val_scores = learning_curve(
        model, X_trainval, y_trainval,
        train_sizes  = train_sizes_pct,
        cv           = cv_lc,
        scoring      = 'roc_auc',
        n_jobs       = -1,
        shuffle      = True,
        random_state = RANDOM_STATE
    )

    tr_mean = train_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    va_mean = val_scores.mean(axis=1)
    va_std  = val_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color=color,
            label='Score Train', linewidth=2, markersize=5)
    ax.fill_between(train_sizes,
                    tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.15, color=color)

    ax.plot(train_sizes, va_mean, 's--', color='gray',
            label='Score Validation', linewidth=2, markersize=5)
    ax.fill_between(train_sizes,
                    va_mean - va_std, va_mean + va_std,
                    alpha=0.10, color='gray')

    ax.axhline(0.5, color='red', linestyle=':', alpha=0.5,
               label='Baseline (aléatoire=0.5)')

    gap = tr_mean[-1] - va_mean[-1]
    verdict = "OK — pas d'overfitting" if gap < 0.05 else "⚠ Overfitting"
    ax.text(0.03, 0.05,
            f"Gap final : {gap:.3f}\n{verdict}",
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='lightyellow', alpha=0.9))

    ax.set_title(f"{name}  (AUC test={results[name]['AUC']:.3f})")
    ax.set_xlabel("Taille du set d'entraînement (lignes)")
    ax.set_ylabel("AUC (ROC)")
    ax.set_ylim(0.45, 0.85)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
path_g1 = os.path.join(OUTPUT_DIR, 'g1_learning_curves.png')
plt.savefig(path_g1, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g1}")


# ── GRAPHE 2 : Matrices de confusion ─────────────────────────────────────────
# Lecture :
#   TP (bas-droite)  : a_corriger correctement détectés ← maximiser
#   FN (bas-gauche)  : a_corriger manqués               ← minimiser (coûteux)
#   FP (haut-droite) : validées bloquées à tort         ← acceptable
#   TN (haut-gauche) : validées correctement identifiées

print("  → Graphe 2/6 : Matrices de Confusion...")

fig2, axes = plt.subplots(2, 2, figsize=(13, 10))
fig2.suptitle(
    "Matrices de Confusion — Set de Test\n"
    "(Seuil optimisé pour Recall ≥ 0.72 sur 'a_corriger')",
    fontsize=13, fontweight='bold'
)

labels_cm = ['validee\n(0)', 'a_corriger\n(1)']

for ax, (name, res) in zip(axes.flatten(), results.items()):
    cm     = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, ax=ax, annot=False, cmap='Blues',
                xticklabels=labels_cm, yticklabels=labels_cm,
                linewidths=0.8, linecolor='white', cbar=False,
                vmin=0, vmax=cm.max())

    for i in range(2):
        for j in range(2):
            txt_color = 'white' if cm_pct[i, j] > 45 else 'black'
            ax.text(j + 0.5, i + 0.38, f'{cm[i, j]}',
                    ha='center', va='center',
                    fontsize=20, fontweight='bold', color=txt_color)
            ax.text(j + 0.5, i + 0.65, f'({cm_pct[i, j]:.1f}%)',
                    ha='center', va='center', fontsize=10, color=txt_color)

    ax.set_title(
        f"{name}  (seuil={res['threshold']:.2f})\n"
        f"AUC={res['AUC']:.3f}  "
        f"Recall={res['Recall']:.3f}  "
        f"F1={res['F1']:.3f}",
        fontsize=10
    )
    ax.set_xlabel('Prédit', fontsize=9)
    ax.set_ylabel('Réel', fontsize=9)

plt.tight_layout()
path_g2 = os.path.join(OUTPUT_DIR, 'g2_confusion_matrices.png')
plt.savefig(path_g2, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g2}")


# ── GRAPHE 3 : Courbes ROC et Precision-Recall ───────────────────────────────

print("  → Graphe 3/6 : Courbes ROC et Precision-Recall...")

fig3, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
fig3.suptitle("Courbes ROC et Précision-Rappel — Set de Test",
              fontsize=13, fontweight='bold')

for name, y_proba in probas_test.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = results[name]['AUC']
    ax_roc.plot(fpr, tpr, color=MODEL_COLORS[name],
                linewidth=2.5, label=f"{name} (AUC={auc:.3f})")
    # Point au seuil optimal
    t   = results[name]['threshold']
    y_t = (y_proba >= t).astype(int)
    fpr_t = ((y_t == 1) & (y_test == 0)).sum() / (y_test == 0).sum()
    ax_roc.scatter(fpr_t, results[name]['Recall'],
                   color=MODEL_COLORS[name], s=120, marker='D',
                   zorder=5, edgecolors='black', linewidths=0.8)

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Baseline')
ax_roc.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax_roc.set_ylabel('Taux de Vrais Positifs (Recall)', fontsize=11)
ax_roc.set_title('Courbe ROC\n(◆ = point au seuil optimal)')
ax_roc.legend(fontsize=9, loc='lower right')
ax_roc.set_xlim([-0.02, 1.02])
ax_roc.set_ylim([-0.02, 1.02])

baseline_pr = float(y_test.mean())
for name, y_proba in probas_test.items():
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_proba)
    ax_pr.plot(rec_arr, prec_arr, color=MODEL_COLORS[name],
               linewidth=2.5, label=name)
    ax_pr.scatter(results[name]['Recall'], results[name]['Precision'],
                  color=MODEL_COLORS[name], s=120, marker='D',
                  zorder=5, edgecolors='black', linewidths=0.8)

ax_pr.axhline(baseline_pr, color='red', linestyle='--', alpha=0.5,
              label=f'Baseline ({baseline_pr:.2f})')
ax_pr.axvline(0.72, color='green', linestyle=':', alpha=0.6,
              label='Recall cible (0.72)')
ax_pr.set_xlabel('Recall', fontsize=11)
ax_pr.set_ylabel('Precision', fontsize=11)
ax_pr.set_title('Courbe Précision-Rappel\n(◆ = point au seuil optimal)')
ax_pr.legend(fontsize=9)
ax_pr.set_xlim([-0.02, 1.05])
ax_pr.set_ylim([0.40, 1.02])

plt.tight_layout()
path_g3 = os.path.join(OUTPUT_DIR, 'g3_roc_pr_curves.png')
plt.savefig(path_g3, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g3}")


# ── GRAPHE 4 : Seuil vs Métriques ────────────────────────────────────────────
# Montre l'évolution de Recall, Precision, F1 selon le seuil choisi.
# Permet de justifier et comprendre le seuil retenu.

print("  → Graphe 4/6 : Courbes Seuil → Métriques...")

fig4, axes = plt.subplots(2, 2, figsize=(14, 10))
fig4.suptitle(
    "Impact du Seuil de Décision sur les Métriques\n"
    "(Calibré sur set de validation | ▼ = seuil retenu)",
    fontsize=13, fontweight='bold'
)

for ax, (name, model) in zip(axes.flatten(), models.items()):
    y_proba_val = model.predict_proba(X_val)[:, 1]
    thresholds  = np.arange(0.20, 0.81, 0.01)
    recs, precs, f1s = [], [], []

    for t in thresholds:
        p = (y_proba_val >= t).astype(int)
        recs.append(recall_score(y_val, p, zero_division=0))
        precs.append(precision_score(y_val, p, zero_division=0))
        f1s.append(f1_score(y_val, p, zero_division=0))

    ax.plot(thresholds, recs,  color='#E53935', linewidth=2,
            label='Recall', linestyle='-')
    ax.plot(thresholds, precs, color='#1E88E5', linewidth=2,
            label='Precision', linestyle='--')
    ax.plot(thresholds, f1s,   color='#43A047', linewidth=2.5,
            label='F1-Score', linestyle='-.')

    best_t = optimal_thrs[name]
    idx    = int(round((best_t - 0.20) / 0.01))
    idx    = max(0, min(idx, len(f1s)-1))
    ax.axvline(best_t, color='black', linestyle=':', linewidth=2, alpha=0.8)
    ax.annotate(
        f' seuil={best_t:.2f}\n F1={f1s[idx]:.3f}',
        xy=(best_t, f1s[idx]),
        fontsize=9, color='black',
        bbox=dict(boxstyle='round,pad=0.2',
                  facecolor='lightyellow', alpha=0.9)
    )
    ax.axhspan(0.72, 0.82, alpha=0.07, color='green',
               label='Zone cible recall')
    ax.axhline(0.72, color='green', linestyle=':', alpha=0.4, linewidth=1)

    ax.set_title(f"{name}")
    ax.set_xlabel("Seuil de décision")
    ax.set_ylabel("Score (sur set de validation)")
    ax.set_ylim([0, 1.05])
    ax.legend(fontsize=8, loc='center left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
path_g4 = os.path.join(OUTPUT_DIR, 'g4_threshold_curves.png')
plt.savefig(path_g4, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g4}")


# ── GRAPHE 5 : Comparaison inter-modèles ─────────────────────────────────────

print("  → Graphe 5/6 : Comparaison des modèles...")

fig5 = plt.figure(figsize=(16, 12))
gs   = gridspec.GridSpec(2, 2, figure=fig5, hspace=0.42, wspace=0.35)

model_names = list(results.keys())
x_pos       = np.arange(len(model_names))
bar_colors  = [MODEL_COLORS[n] for n in model_names]

# Barres groupées : AUC / Recall / F1 / Precision / Accuracy
ax_bar = fig5.add_subplot(gs[0, :])
metrics_plot  = ['AUC', 'Recall', 'F1', 'Precision', 'Accuracy']
metric_colors = ['#2196F3', '#E53935', '#43A047', '#FF9800', '#9C27B0']
bar_w = 0.15
offsets = np.linspace(
    -(len(metrics_plot)-1)/2,
     (len(metrics_plot)-1)/2,
     len(metrics_plot)
) * bar_w

for i, (m, mc) in enumerate(zip(metrics_plot, metric_colors)):
    vals = [results[n][m] for n in model_names]
    bars = ax_bar.bar(x_pos + offsets[i], vals,
                      width=bar_w, label=m, color=mc,
                      alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom',
                    fontsize=7, rotation=45)

ax_bar.axhline(0.72, color='red', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible Recall (0.72)')
ax_bar.axhline(0.68, color='green', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible F1 (0.68)')
ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(model_names, fontsize=11)
ax_bar.set_ylabel('Score', fontsize=11)
ax_bar.set_title(
    'Comparaison des Métriques par Modèle — Set de Test\n'
    '(features originales uniquement, seuil optimisé)',
    fontsize=12, fontweight='bold'
)
ax_bar.set_ylim(0, 1.18)
ax_bar.legend(fontsize=9, loc='upper right', ncol=4)
ax_bar.set_facecolor('#fafafa')
for tick, name in zip(ax_bar.get_xticklabels(), model_names):
    tick.set_color(MODEL_COLORS[name])
    tick.set_fontweight('bold')

# TP / FN / TN / FP empilés
ax_cm = fig5.add_subplot(gs[1, 0])
tp_v = [results[n]['TP'] for n in model_names]
fn_v = [results[n]['FN'] for n in model_names]
tn_v = [results[n]['TN'] for n in model_names]
fp_v = [results[n]['FP'] for n in model_names]

bw = 0.5
ax_cm.bar(x_pos, tp_v, bw, label='TP (a_corriger détectés)',
          color='#43A047', alpha=0.9)
ax_cm.bar(x_pos, fn_v, bw, bottom=tp_v,
          label='FN (a_corriger manqués)', color='#E53935', alpha=0.9)
ax_cm.bar(x_pos, tn_v, bw,
          bottom=[tp_v[i]+fn_v[i] for i in range(len(model_names))],
          label='TN (validées OK)', color='#1E88E5', alpha=0.9)
ax_cm.bar(x_pos, fp_v, bw,
          bottom=[tp_v[i]+fn_v[i]+tn_v[i] for i in range(len(model_names))],
          label='FP (validées bloquées)', color='#FF9800', alpha=0.9)
for i, name in enumerate(model_names):
    ax_cm.text(i, tp_v[i]/2, f"TP\n{tp_v[i]}",
               ha='center', va='center', fontsize=9,
               color='white', fontweight='bold')
    ax_cm.text(i, tp_v[i] + fn_v[i]/2, f"FN\n{fn_v[i]}",
               ha='center', va='center', fontsize=9,
               color='white', fontweight='bold')
ax_cm.set_xticks(x_pos)
ax_cm.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_cm.set_ylabel('Nombre de prédictions')
ax_cm.set_title(f'Décomposition TP/FN/TN/FP\n(Test set = {len(y_test)} lignes)',
                fontsize=11, fontweight='bold')
ax_cm.legend(fontsize=7, loc='upper right')
ax_cm.set_facecolor('#fafafa')

# Seuil retenu par modèle
ax_thr = fig5.add_subplot(gs[1, 1])
thr_v = [results[n]['threshold'] for n in model_names]
bars  = ax_thr.bar(x_pos, thr_v, 0.5,
                    color=bar_colors, alpha=0.85,
                    edgecolor='white', linewidth=0.8)
ax_thr.axhline(0.5, color='black', linestyle='--', alpha=0.4,
               label='Seuil par défaut (0.5)')
for bar, t in zip(bars, thr_v):
    ax_thr.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{t:.2f}', ha='center', va='bottom',
                fontsize=13, fontweight='bold')
ax_thr.set_xticks(x_pos)
ax_thr.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_thr.set_ylabel('Seuil de décision optimal')
ax_thr.set_title('Seuil Optimal par Modèle\n(calibré sur set de validation)',
                  fontsize=11, fontweight='bold')
ax_thr.set_ylim(0, 0.80)
ax_thr.legend(fontsize=9)
ax_thr.set_facecolor('#fafafa')

plt.suptitle('Synthèse Comparative — Pipeline ML (Features originales)',
             fontsize=14, fontweight='bold', y=1.01)
path_g5 = os.path.join(OUTPUT_DIR, 'g5_model_comparison.png')
plt.savefig(path_g5, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g5}")


# ── GRAPHE 6 : Importance des features (GBM + RF) ────────────────────────────

print("  → Graphe 6/6 : Importance des features...")

fig6, axes = plt.subplots(1, 2, figsize=(14, 6))
fig6.suptitle(
    "Importance des Features — GradientBoosting et RandomForest\n"
    "(Gini importance : contribution de chaque feature à la réduction d'impureté)",
    fontsize=13, fontweight='bold'
)

for ax, name in zip(axes, ['GradientBoosting', 'RandomForest']):
    clf        = models[name].named_steps['clf']
    imp        = clf.feature_importances_
    indices    = np.argsort(imp)[::-1]
    feat_names = [FEATURES_RETENUES[i] for i in indices]
    feat_imp   = imp[indices]

    colors_fi = [MODEL_COLORS[name]] * len(FEATURES_RETENUES)

    ax.barh(range(len(FEATURES_RETENUES)), feat_imp,
            color=colors_fi, alpha=0.85,
            edgecolor='white', linewidth=0.5)

    # Valeur sur chaque barre
    for j, (val, fname) in enumerate(zip(feat_imp, feat_names)):
        ax.text(val + 0.003, j, f'{val:.4f}',
                va='center', fontsize=9)

    ax.set_yticks(range(len(FEATURES_RETENUES)))
    ax.set_yticklabels(feat_names, fontsize=10)
    ax.set_xlabel("Importance (Gini)", fontsize=10)
    ax.set_title(f"{name}", fontsize=11)
    ax.invert_yaxis()
    ax.set_facecolor('#fafafa')
    ax.set_xlim(0, feat_imp.max() * 1.25)

plt.tight_layout()
path_g6 = os.path.join(OUTPUT_DIR, 'g6_feature_importance.png')
plt.savefig(path_g6, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g6}")


# ──────────────────────────────────────────────────────────────────────────────
# RAPPORT FINAL — RÉSULTATS DÉTAILLÉS
# ──────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  RÉSULTATS DÉTAILLÉS — SET DE TEST")
print("=" * 70)

CIBLE_AUC    = (0.68, 0.75)
CIBLE_RECALL = (0.72, 0.80)
CIBLE_F1     = (0.68, 0.75)

for name, res in results.items():
    auc_ok = "✓ CIBLE" if CIBLE_AUC[0] <= res['AUC'] else (
              "↗ AU-DESSUS" if res['AUC'] > CIBLE_AUC[1] else "✗ SOUS CIBLE")
    rec_ok = "✓ CIBLE" if CIBLE_RECALL[0] <= res['Recall'] <= CIBLE_RECALL[1] else (
              "↗ AU-DESSUS" if res['Recall'] > CIBLE_RECALL[1] else "✗ SOUS CIBLE")
    f1_ok  = "✓ CIBLE" if CIBLE_F1[0] <= res['F1'] <= CIBLE_F1[1] else (
              "↗ AU-DESSUS" if res['F1'] > CIBLE_F1[1] else "✗ SOUS CIBLE")

    print(f"\n  ┌─ {name} {'─'*(44-len(name))}")
    print(f"  │  Seuil optimal        : {res['threshold']:.2f}")
    print(f"  │  AUC                  : {res['AUC']:.4f}   {auc_ok}")
    print(f"  │  Accuracy             : {res['Accuracy']:.4f}")
    print(f"  │  Precision            : {res['Precision']:.4f}")
    print(f"  │  Recall (a_corriger)  : {res['Recall']:.4f}   {rec_ok}")
    print(f"  │  F1    (a_corriger)   : {res['F1']:.4f}   {f1_ok}")
    print(f"  │  Specificity          : {res['Specificity']:.4f}")
    print(f"  │  Matrice confusion    : TP={res['TP']}  FP={res['FP']}  "
          f"FN={res['FN']}  TN={res['TN']}")
    print(f"  └──────────────────────────────────────────────────")

    print(f"\n  Classification Report complet — {name} :")
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['validee', 'a_corriger'],
        digits=4
    ))

# Tableau récapitulatif
print("=" * 80)
print("  TABLEAU RÉCAPITULATIF — SET DE TEST")
print("=" * 80)
metrics_order = ['threshold', 'AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity']
hdr = f"  {'Modèle':<22}" + "".join(f"{m:>12}" for m in metrics_order)
print(hdr)
print("  " + "─" * 78)
for name, res in results.items():
    row = f"  {name:<22}"
    for m in metrics_order:
        row += f"{res[m]:>12.4f}"
    print(row)
print("  " + "─" * 78)

best_row = f"  {'MEILLEUR':<22}"
best_row += "            —"
for m in metrics_order[1:]:
    best_val   = max(results[n][m] for n in results)
    best_row  += f"{best_val:>12.4f}"
print(best_row)
print("=" * 80)

# Recommandation finale
scores_comb = {
    n: results[n]['F1']*0.4 + results[n]['Recall']*0.4 + results[n]['AUC']*0.2
    for n in results
}
best = max(scores_comb, key=scores_comb.get)

print("\n  RECOMMANDATION FINALE")
print("  Score composite = 0.4×F1 + 0.4×Recall + 0.2×AUC")
print()
for name, sc in sorted(scores_comb.items(), key=lambda x: -x[1]):
    flag = "  ← RECOMMANDÉ" if name == best else ""
    print(f"    {name:<22} score={sc:.4f}{flag}")

print(f"\n  ➤ Modèle recommandé   : {best}")
print(f"  ➤ Seuil               : {results[best]['threshold']:.2f}")
print(f"  ➤ AUC                 : {results[best]['AUC']:.4f}")
print(f"  ➤ Recall (a_corriger) : {results[best]['Recall']:.4f}")
print(f"  ➤ F1    (a_corriger)  : {results[best]['F1']:.4f}")

print("\n" + "=" * 70)
print("  IMAGES GÉNÉRÉES")
print("=" * 70)
generated = [path_g1, path_g2, path_g3, path_g4, path_g5, path_g6]
labels_g  = [
    "Courbes d'apprentissage",
    "Matrices de confusion",
    "Courbes ROC et Précision-Rappel",
    "Seuil → Métriques",
    "Comparaison inter-modèles",
    "Importance des features",
]
for path, label in zip(generated, labels_g):
    size_kb = os.path.getsize(path) // 1024
    print(f"  ✓ {label:<35} → {os.path.basename(path)} ({size_kb} Ko)")

print("\n" + "=" * 70)
print("  NOTE PLAFOND THÉORIQUE")
print("=" * 70)
print("""
  Avec les features originales uniquement :
  → 90% des lignes ont les mêmes features binaires mais des labels différents
  → Bayes Error Rate = 35.8%  → Plafond accuracy = 64.2%
  → Aucun algorithme ne peut dépasser ce plafond de façon robuste

  Pour atteindre 80%+ d'accuracy, il faudrait enrichir avec :
    1. Les valeurs numériques brutes (quantites_actes, couts réels)
    2. L'historique du prescripteur (taux d'erreur par id_prescripteur)
    3. Des features temporelles (délais entre dates)

  Référence scientifique :
    Fukunaga, K. (1990). Introduction to Statistical Pattern Recognition.
    Academic Press. Chapter 3 — Bayes Error and Its Estimation.
""")

# ==============================================================================
# ÉCRITURE DU RAPPORT DANS LE FICHIER
# ==============================================================================

sys.stdout = tee.stdout    # Restaurer stdout
log_content = tee.getvalue()

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(log_content)

print(f"\n  ✓ Rapport complet enregistré : {REPORT_PATH}")
print(f"    Taille : {os.path.getsize(REPORT_PATH) // 1024} Ko")
print("\nPipeline terminé avec succès.")


   PIPELINE ML — VÉRIFICATION FACTURES (Features originales)

[ÉTAPE 1/6] Chargement des données
  ➤ Fichier             : data/dfforml2.csv
  ➤ Lignes totales      : 1433
  ➤ Colonnes totales    : 40
  ➤ Classe 'validee'    : 600 (41.9%)
  ➤ Classe 'a_corriger' : 833 (58.1%)
  ➤ Ratio déséquilibre  : 1.39

  Analyse de variance des colonnes originales :
  Colonne                                        Dominance  Statut
  ─────────────────────────────────────────────────────────────────
  consultation_type                                  0.404  ✓ UTILE
  type_prestation                                    0.376  ✓ UTILE
  mode_sortie                                        0.996  ✗ MORTE (constante >97%)
  nom_patient_is_filled                              0.999  ✗ MORTE (constante >97%)
  distance_village_is_filled                         1.000  ✗ MORTE (constante >97%)
  age_patient_is_filled                              1.000  ✗ MORTE (constante >97%)
  registre_number_is_filled     